# SMARTADDICT v4 — OOF SIGNAL FORGE (improved)

Incremental upgrade over the v1 challenger. What changed:

1. **Model diversity** — adds `HGB` (sklearn HistGradientBoosting) and `LGB_Deep` (high-leaves, low-lr LightGBM) to the core `LGB / XGB / CAT` trio.
2. **Seed bagging** — each core model is averaged over multiple seeds to cut OOF variance (`SEED_BAG`).
3. **Richer features** — `known_usage_hours`, `unaccounted_screen_time`, `total_screen_week`, `weekend_share`, `avg_session_min`, `notifs_per_app_open`, log-bounded transforms, more interactions and flags.
4. **Stronger blend** — nested OOF selection now searches `prob`, `rank`, `scipy` (Nelder-Mead weights) and the `prob+rank` average.

All models share the same 5 stratified folds. Every reported score is honest OOF.


In [ ]:
# 1. Setup
import os, re, gc, time, random, warnings, hashlib
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score
from scipy.stats import rankdata
from scipy.optimize import minimize
from sklearn.ensemble import HistGradientBoostingClassifier

from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")

SEED = 20260807
N_FOLDS = 5
EARLY_STOP = 150
FAST_MODE = False   # True = smoke-test only; False = competition run

# Seeds per model (seed bagging = variance reduction).
SEED_BAG = {"LGB": 2, "XGB": 2, "HGB": 2, "CAT": 1, "LGB_Deep": 1}

random.seed(SEED)
np.random.seed(SEED)

if FAST_MODE:
    N_FOLDS = 2
    EARLY_STOP = 50
    SEED_BAG = {k: 1 for k in SEED_BAG}

def banner(text):
    print("\n" + "=" * 72)
    print(text)
    print("=" * 72)

banner("SMARTADDICT V4 — OOF SIGNAL FORGE")
print(f"Seed        : {SEED}")
print(f"Folds       : {N_FOLDS}")
print(f"FAST_MODE   : {FAST_MODE}")
print(f"Seed bag    : {SEED_BAG}")
print("Validation  : true stratified OOF + nested blend selection")
print("Training    : deterministic CPU configuration")


In [ ]:
# 2. Robust competition-file autodiscovery
def _csv_header(path):
    try:
        return list(pd.read_csv(path, nrows=0).columns)
    except Exception:
        return None

def _valid_competition_trio(train_path, test_path, sample_path):
    tr_cols = _csv_header(train_path)
    te_cols = _csv_header(test_path)
    sub_cols = _csv_header(sample_path)
    if not tr_cols or not te_cols or not sub_cols:
        return False, None
    target_candidates = [c for c in tr_cols if c not in te_cols]
    if len(target_candidates) != 1:
        return False, None
    target = target_candidates[0]
    if not set(te_cols).issubset(set(tr_cols)):
        return False, None
    if target not in sub_cols or len(sub_cols) < 2:
        return False, None
    shared_keys = [c for c in sub_cols if c in te_cols and c != target]
    if not shared_keys:
        return False, None
    return True, target

def _candidate_trios():
    seen = set()
    canonical_dirs = [
        Path('/kaggle/input/playground-series-s6e8'),
        Path('/kaggle/input/playground-series-season-6-episode-8'),
    ]
    for d in canonical_dirs:
        trio = (d/'train.csv', d/'test.csv', d/'sample_submission.csv')
        if all(p.exists() for p in trio):
            key = tuple(map(str, trio))
            if key not in seen:
                seen.add(key); yield trio

    kaggle_root = Path('/kaggle/input')
    if kaggle_root.exists():
        for train_path in kaggle_root.rglob('train.csv'):
            d = train_path.parent
            trio = (train_path, d/'test.csv', d/'sample_submission.csv')
            if all(p.exists() for p in trio):
                key = tuple(map(str, trio))
                if key not in seen:
                    seen.add(key); yield trio

    # Local fallbacks: current dir, datasets/, /mnt/data.
    for root in [Path('.'), Path('datasets'), Path('/mnt/data')]:
        if not root.exists():
            continue
        train_files = sorted(root.glob('train*.csv'))
        test_files = sorted(root.glob('test*.csv'))
        sample_files = sorted(root.glob('sample_submission*.csv'))
        for tr in train_files:
            for te in test_files:
                for sub in sample_files:
                    trio = (tr, te, sub)
                    key = tuple(map(str, trio))
                    if key not in seen:
                        seen.add(key); yield trio

def locate_competition_files():
    valid = []
    for tr, te, sub in _candidate_trios():
        ok, target = _valid_competition_trio(tr, te, sub)
        if not ok:
            continue
        path_text = str(tr.parent).lower()
        keyword_score = sum(token in path_text for token in ['playground', 's6e8', 'season-6', 'smartphone', 'addiction'])
        size_score = tr.stat().st_size if tr.exists() else 0
        valid.append((keyword_score, size_score, tr, te, sub, target))
    if not valid:
        raise FileNotFoundError(
            'No schema-compatible competition train/test/sample_submission trio found. '
            'In Kaggle, attach the competition data under Add Input.'
        )
    valid.sort(key=lambda x: (x[0], x[1]), reverse=True)
    _, _, train_path, test_path, sample_path, target = valid[0]
    return train_path, test_path, sample_path, target

TRAIN_PATH, TEST_PATH, SAMPLE_PATH, TARGET = locate_competition_files()
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)
ID_COL = 'id' if 'id' in train.columns else sample.columns[0]

assert TARGET in train.columns and TARGET not in test.columns
assert set(test.columns).issubset(set(train.columns))
assert train[ID_COL].is_unique
assert test[ID_COL].is_unique
assert train[TARGET].isin([0, 1]).all()
assert len(test) == len(sample)
assert ID_COL in sample.columns and TARGET in sample.columns

banner('DATA LOADED')
print(f'Train path  : {TRAIN_PATH}')
print(f'Test path   : {TEST_PATH}')
print(f'Train       : {train.shape}')
print(f'Test        : {test.shape}')
print(f'Target      : {TARGET}')
print(f'Positive    : {train[TARGET].mean():.4%}')


In [ ]:
# 3. Targeted integrity / EDA
feature_cols_raw = [c for c in test.columns if c != ID_COL]
cat_raw = [c for c in feature_cols_raw if train[c].dtype == "object"]
num_raw = [c for c in feature_cols_raw if c not in cat_raw]

missing = (train[feature_cols_raw].isna().mean() * 100).sort_values(ascending=False)

banner("QUICK SIGNAL AUDIT")
print(f"Numeric features      : {len(num_raw)}")
print(f"Categorical features  : {len(cat_raw)}")
print(f"Missing cells         : {int(train[feature_cols_raw].isna().sum().sum()):,}")
print("\nHighest missing rates:")
print(missing.head(8).round(2).astype(str).add("%").to_string())

print("\nCategories:")
for c in cat_raw:
    vals = sorted(train[c].dropna().astype(str).unique().tolist())
    print(f"  * {c}: {vals} + MISSING")

raw_auc = {}
for c in num_raw:
    tmp = train[[c, TARGET]].dropna()
    if tmp[c].nunique() > 1:
        a = roc_auc_score(tmp[TARGET], tmp[c])
        raw_auc[c] = max(a, 1 - a)

print("\nStrongest raw univariate AUC signals:")
for c, a in sorted(raw_auc.items(), key=lambda x: x[1], reverse=True)[:6]:
    print(f"  * {c:<28} {a:.5f}")


In [ ]:
# 4. Deterministic feature engineering (expanded)
BASE_CAT = ["gender", "stress_level", "academic_work_impact"]

def _safe_ratio(a, b, eps=1e-3):
    return a / (b + eps)

def _fixed_bin(series, bins, labels):
    out = pd.cut(series, bins=bins, labels=labels, include_lowest=True)
    return out.astype("string").fillna("__MISSING__").astype(str)

def engineer_features(df):
    x = df.copy()
    if ID_COL in x.columns:
        x = x.drop(columns=[ID_COL])

    # Normalize base categoricals.
    for c in BASE_CAT:
        if c in x.columns:
            x[c] = x[c].astype("string").fillna("__MISSING__").astype(str)

    base_for_missing = [c for c in feature_cols_raw if c in x.columns]
    x["missing_count"] = x[base_for_missing].isna().sum(axis=1).astype("int8")
    for c in base_for_missing:
        x[f"miss__{c}"] = x[c].isna().astype("int8") if c not in BASE_CAT else (x[c] == "__MISSING__").astype("int8")

    # Behavioral load.
    x["social_gaming_hours"] = x["social_media_hours"] + x["gaming_hours"]
    x["known_usage_hours"] = x["social_media_hours"] + x["gaming_hours"] + x["work_study_hours"]
    x["unaccounted_screen_time"] = x["daily_screen_time_hours"] - x["known_usage_hours"]
    x["digital_productive_gap"] = x["social_gaming_hours"] - x["work_study_hours"]
    x["engagement_load"] = (
        x["daily_screen_time_hours"]
        + 0.50 * x["weekend_screen_time"]
        + x["social_media_hours"]
        + x["gaming_hours"]
    )
    x["total_screen_week"] = x["daily_screen_time_hours"] * 5 + x["weekend_screen_time"] * 2
    x["weekend_share"] = _safe_ratio(x["weekend_screen_time"] * 2, x["total_screen_week"])

    # Weekend shift.
    x["screen_gap_weekend"] = x["weekend_screen_time"] - x["daily_screen_time_hours"]
    x["weekend_weekday_ratio"] = _safe_ratio(x["weekend_screen_time"], x["daily_screen_time_hours"])

    # Composition.
    x["social_share_screen"] = _safe_ratio(x["social_media_hours"], x["daily_screen_time_hours"])
    x["gaming_share_screen"] = _safe_ratio(x["gaming_hours"], x["daily_screen_time_hours"])
    x["productive_share_screen"] = _safe_ratio(x["work_study_hours"], x["daily_screen_time_hours"])
    x["digital_productive_ratio"] = _safe_ratio(x["social_gaming_hours"], x["work_study_hours"])
    x["social_share_leisure"] = _safe_ratio(x["social_media_hours"], x["social_gaming_hours"])
    x["gaming_share_leisure"] = _safe_ratio(x["gaming_hours"], x["social_gaming_hours"])

    # Attention intensity.
    x["opens_per_screen_hour"] = _safe_ratio(x["app_opens_per_day"], x["daily_screen_time_hours"])
    x["notifs_per_screen_hour"] = _safe_ratio(x["notifications_per_day"], x["daily_screen_time_hours"])
    x["opens_per_notification"] = x["app_opens_per_day"] / (x["notifications_per_day"] + 1.0)
    x["notifs_per_app_open"] = _safe_ratio(x["notifications_per_day"], x["app_opens_per_day"])
    x["avg_session_min"] = _safe_ratio(x["daily_screen_time_hours"] * 60.0, x["app_opens_per_day"])
    x["interaction_intensity"] = np.sqrt(
        np.clip(x["app_opens_per_day"], 0, None) * np.clip(x["notifications_per_day"], 0, None)
    )

    # Recovery / sleep pressure.
    x["sleep_deficit_8h"] = 8.0 - x["sleep_hours"]
    x["screen_sleep_ratio"] = _safe_ratio(x["daily_screen_time_hours"], x["sleep_hours"])
    x["weekend_sleep_ratio"] = _safe_ratio(x["weekend_screen_time"], x["sleep_hours"])
    x["social_sleep_ratio"] = _safe_ratio(x["social_media_hours"], x["sleep_hours"])
    x["gaming_sleep_ratio"] = _safe_ratio(x["gaming_hours"], x["sleep_hours"])
    x["screen_minus_sleep"] = x["daily_screen_time_hours"] - x["sleep_hours"]
    x["awake_hours"] = 24.0 - x["sleep_hours"]
    x["notifs_per_awake_hour"] = _safe_ratio(x["notifications_per_day"], x["awake_hours"])
    x["opens_per_awake_hour"] = _safe_ratio(x["app_opens_per_day"], x["awake_hours"])

    # Explicit interactions.
    x["screen_x_social"] = x["daily_screen_time_hours"] * x["social_media_hours"]
    x["screen_x_gaming"] = x["daily_screen_time_hours"] * x["gaming_hours"]
    x["screen_x_weekend"] = x["daily_screen_time_hours"] * x["weekend_screen_time"]
    x["social_x_gaming"] = x["social_media_hours"] * x["gaming_hours"]
    x["screen_x_app_opens"] = x["daily_screen_time_hours"] * x["app_opens_per_day"]
    x["screen_x_notifs"] = x["daily_screen_time_hours"] * x["notifications_per_day"]
    x["age_x_screen"] = x["age"] * x["daily_screen_time_hours"]
    x["age_x_social"] = x["age"] * x["social_media_hours"]
    x["age_x_weekend"] = x["age"] * x["weekend_screen_time"]

    # Log-bounded transforms (heavy-tail control).
    x["log1p_notifs"] = np.log1p(x["notifications_per_day"])
    x["log1p_opens"] = np.log1p(x["app_opens_per_day"])
    x["log1p_screen"] = np.log1p(x["daily_screen_time_hours"])

    # Flags.
    x["flag_high_screen"] = (x["daily_screen_time_hours"] >= 8).astype("int8")
    x["flag_short_sleep"] = (x["sleep_hours"] < 7).astype("int8")
    x["flag_high_notifs"] = (x["notifications_per_day"] >= 100).astype("int8")

    # Fixed, target-independent behavioral bins.
    x["age_band"] = _fixed_bin(x["age"], [0, 18, 21, 25, 30, 120], ["<=18", "19-21", "22-25", "26-30", "31+"])
    x["screen_band"] = _fixed_bin(x["daily_screen_time_hours"], [-np.inf, 4, 6, 8, 10, np.inf], ["<=4", "4-6", "6-8", "8-10", "10+"])
    x["sleep_band"] = _fixed_bin(x["sleep_hours"], [-np.inf, 5.5, 6.5, 7.5, 8.5, np.inf], ["<5.5", "5.5-6.5", "6.5-7.5", "7.5-8.5", "8.5+"])

    # Low-cardinality crosses.
    x["stress_x_impact"] = x["stress_level"] + "__" + x["academic_work_impact"]
    x["gender_x_stress"] = x["gender"] + "__" + x["stress_level"]
    x["gender_x_impact"] = x["gender"] + "__" + x["academic_work_impact"]

    return x

X_train = engineer_features(train.drop(columns=[TARGET]))
X_test = engineer_features(test)

CAT_COLS = [c for c in X_train.columns if X_train[c].dtype == "object"]
NUM_COLS = [c for c in X_train.columns if c not in CAT_COLS]

assert list(X_train.columns) == list(X_test.columns)

banner("FEATURE FORGE")
print(f"Raw predictors       : {len(feature_cols_raw)}")
print(f"Engineered features  : {X_train.shape[1]}")
print(f"Categorical          : {len(CAT_COLS)}")
print(f"Numeric              : {len(NUM_COLS)}")


## Optional original-data severity teacher
This branch is unchanged from v1. It is skipped automatically when the public `smart-phone` dataset is not mounted (e.g. this local run). If present, it injects a CatBoost severity-teacher OOF feature family into the frames and enables the `LGB_EXT` challenger.


In [ ]:
# 5. Optional original-data branch
def snake_case(name):
    s = re.sub(r"[^0-9a-zA-Z]+", "_", str(name)).strip("_").lower()
    return s

def locate_original_dataset():
    roots = [Path("/kaggle/input")]
    required = {"daily_screen_time_hours", "social_media_hours", "addiction_level", "addicted_label"}
    comp_paths = {str(TRAIN_PATH), str(TEST_PATH), str(SAMPLE_PATH)}

    for root in roots:
        if not root.exists():
            continue
        for p in root.rglob("*.csv"):
            if str(p) in comp_paths:
                continue
            try:
                probe = pd.read_csv(p, nrows=3, keep_default_na=False)
                cols = {snake_case(c) for c in probe.columns}
                if required.issubset(cols):
                    return p
            except Exception:
                pass
    return None

ORIGINAL_PATH = locate_original_dataset()
HAS_ORIGINAL = ORIGINAL_PATH is not None

teacher_feature_names = []
X_original = None
y_original = None

if HAS_ORIGINAL:
    original = pd.read_csv(ORIGINAL_PATH, keep_default_na=False)
    original.columns = [snake_case(c) for c in original.columns]

    for c in num_raw:
        if c in original.columns:
            original[c] = pd.to_numeric(original[c], errors="coerce")

    for c in BASE_CAT:
        if c in original.columns:
            original[c] = original[c].replace("", np.nan)

    severity_map = {"none": 0, "mild": 1, "moderate": 2, "severe": 3}
    sev_text = original["addiction_level"].astype(str).str.strip().str.lower()
    valid_sev = sev_text.isin(severity_map)

    original = original.loc[valid_sev].reset_index(drop=True)
    y_severity = sev_text.loc[valid_sev].map(severity_map).astype(int).to_numpy()

    needed = feature_cols_raw
    missing_cols = [c for c in needed if c not in original.columns]
    if missing_cols:
        raise ValueError(f"Original dataset missing expected columns: {missing_cols}")

    original_raw = original[needed].copy()
    y_original = pd.to_numeric(original[TARGET], errors="coerce").astype(int).to_numpy()

    def teacher_frame(df):
        z = df[needed].copy()
        for c in BASE_CAT:
            z[c] = z[c].astype("string").fillna("__MISSING__").astype(str)
        return z

    T_orig = teacher_frame(original_raw)
    T_comp_train = teacher_frame(train[needed])
    T_comp_test = teacher_frame(test[needed])

    sev_oof = np.zeros((len(T_orig), 4), dtype=np.float32)
    sev_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + 17)

    banner("SEVERITY TEACHER")
    for sf, (sti, svi) in enumerate(sev_skf.split(T_orig, y_severity), 1):
        teacher = CatBoostClassifier(
            iterations=500 if not FAST_MODE else 120,
            depth=6, learning_rate=0.04, loss_function="MultiClass",
            random_seed=SEED + sf, l2_leaf_reg=5.0, random_strength=0.20,
            thread_count=-1, verbose=False, allow_writing_files=False,
        )
        teacher.fit(
            T_orig.iloc[sti], y_severity[sti],
            cat_features=BASE_CAT,
            eval_set=(T_orig.iloc[svi], y_severity[svi]),
            use_best_model=True, early_stopping_rounds=60 if not FAST_MODE else 20,
            verbose=False,
        )
        sev_oof[svi] = teacher.predict_proba(T_orig.iloc[svi]).astype(np.float32)

    sev_pred_class = sev_oof.argmax(axis=1)
    teacher_acc = accuracy_score(y_severity, sev_pred_class)

    teacher_full = CatBoostClassifier(
        iterations=500 if not FAST_MODE else 120,
        depth=6, learning_rate=0.04, loss_function="MultiClass",
        random_seed=SEED, l2_leaf_reg=5.0, random_strength=0.20,
        thread_count=-1, verbose=False, allow_writing_files=False,
    )
    teacher_full.fit(T_orig, y_severity, cat_features=BASE_CAT, verbose=False)

    sev_comp_train = teacher_full.predict_proba(T_comp_train).astype(np.float32)
    sev_comp_test = teacher_full.predict_proba(T_comp_test).astype(np.float32)

    teacher_feature_names = [f"teacher_severity_p{k}" for k in range(4)] + ["teacher_expected_severity"]

    for k in range(4):
        X_train[f"teacher_severity_p{k}"] = sev_comp_train[:, k]
        X_test[f"teacher_severity_p{k}"] = sev_comp_test[:, k]

    X_train["teacher_expected_severity"] = sev_comp_train @ np.arange(4, dtype=np.float32)
    X_test["teacher_expected_severity"] = sev_comp_test @ np.arange(4, dtype=np.float32)

    X_original = engineer_features(original_raw)
    for k in range(4):
        X_original[f"teacher_severity_p{k}"] = sev_oof[:, k]
    X_original["teacher_expected_severity"] = sev_oof @ np.arange(4, dtype=np.float32)

    CAT_COLS = [c for c in X_train.columns if X_train[c].dtype == "object"]
    NUM_COLS = [c for c in X_train.columns if c not in CAT_COLS]

    print(f"Original source       : {ORIGINAL_PATH}")
    print(f"Original rows         : {len(original):,}")
    print(f"Teacher OOF accuracy  : {teacher_acc:.5f}  [original severity task]")
    print(f"Teacher features      : {len(teacher_feature_names)}")
else:
    banner("SEVERITY TEACHER")
    print("Original public dataset not mounted.")
    print("Teacher + LGB_EXT branches skipped; core challenger fully executable.")

assert list(X_train.columns) == list(X_test.columns)
if HAS_ORIGINAL:
    assert list(X_original.columns) == list(X_train.columns)


In [ ]:
# 6. Deterministic preprocessing
def make_lgb_frames(train_df, test_df, original_df=None):
    tr = train_df.copy()
    te = test_df.copy()
    og = original_df.copy() if original_df is not None else None

    for c in CAT_COLS:
        levels = sorted(tr[c].astype(str).unique().tolist())
        mapping = {v: i for i, v in enumerate(levels)}
        tr[c] = tr[c].astype(str).map(mapping).fillna(-1).astype("int16")
        te[c] = te[c].astype(str).map(mapping).fillna(-1).astype("int16")
        if og is not None:
            og[c] = og[c].astype(str).map(mapping).fillna(-1).astype("int16")

    for c in NUM_COLS:
        tr[c] = pd.to_numeric(tr[c], errors="coerce").astype("float32")
        te[c] = pd.to_numeric(te[c], errors="coerce").astype("float32")
        if og is not None:
            og[c] = pd.to_numeric(og[c], errors="coerce").astype("float32")

    return tr, te, og

def make_xgb_frames(train_df, test_df):
    # Category universe comes from competition TRAIN only.
    tr = train_df.copy()
    te = test_df.copy()

    for c in CAT_COLS:
        levels = sorted(tr[c].astype(str).unique().tolist())
        tr[c] = pd.Categorical(tr[c].astype(str), categories=levels)
        te[c] = pd.Categorical(te[c].astype(str), categories=levels)

    tr = pd.get_dummies(tr, columns=CAT_COLS, dtype=np.int8)
    te = pd.get_dummies(te, columns=CAT_COLS, dtype=np.int8)
    te = te.reindex(columns=tr.columns, fill_value=0)

    safe_names = {
        c: re.sub(r"[^0-9A-Za-z_]+", "_", str(c))
        for c in tr.columns
    }
    tr = tr.rename(columns=safe_names)
    te = te.rename(columns=safe_names)

    for c in tr.columns:
        if tr[c].dtype not in [np.int8, np.int16, np.int32, np.int64]:
            tr[c] = pd.to_numeric(tr[c], errors="coerce").astype("float32")
            te[c] = pd.to_numeric(te[c], errors="coerce").astype("float32")

    return tr, te

LGB_TRAIN, LGB_TEST, LGB_ORIG = make_lgb_frames(X_train, X_test, X_original)
XGB_TRAIN, XGB_TEST = make_xgb_frames(X_train, X_test)

# CatBoost categorical values are already strings.
CAT_TRAIN = X_train.copy()
CAT_TEST = X_test.copy()

banner("MODEL MATRICES")
print(f"LGB features : {LGB_TRAIN.shape[1]}")
print(f"XGB features : {XGB_TRAIN.shape[1]}")
print(f"CAT features : {CAT_TRAIN.shape[1]}")
print(f"LGB_EXT      : {'enabled' if HAS_ORIGINAL else 'disabled'}")
print(f"Models       : LGB / XGB / CAT / HGB / LGB_Deep")


In [ ]:
# 7. True OOF training (all models, identical stratified folds)
y = train[TARGET].to_numpy(dtype=np.int8)
n_train, n_test = len(train), len(test)

MODEL_NAMES = ["LGB", "XGB", "CAT", "HGB", "LGB_Deep"] + (["LGB_EXT"] if HAS_ORIGINAL else [])
oof = {m: np.zeros(n_train, dtype=np.float32) for m in MODEL_NAMES}
test_fold_preds = {m: np.zeros((N_FOLDS, n_test), dtype=np.float32) for m in MODEL_NAMES}
fold_id = np.full(n_train, -1, dtype=np.int8)

importance_rows = []
fit_start = time.time()

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(n_train), y), 1):
    fold_id[va_idx] = fold - 1
    banner(f"FOLD {fold}/{N_FOLDS}")

    for mname in MODEL_NAMES:
        n_seeds = SEED_BAG.get(mname, 1)
        seed_oof = np.zeros((n_seeds, len(va_idx)), dtype=np.float32)
        seed_test = np.zeros((n_seeds, n_test), dtype=np.float32)
        iter_info = []

        for si in range(n_seeds):
            rnd = SEED + fold + 5000 * si

            if mname == "LGB":
                model = LGBMClassifier(
                    objective="binary", metric="auc",
                    n_estimators=2200 if not FAST_MODE else 250,
                    learning_rate=0.03, num_leaves=63, max_depth=-1,
                    min_child_samples=80, subsample=0.90, subsample_freq=1,
                    colsample_bytree=0.90, reg_alpha=0.15, reg_lambda=1.50,
                    max_bin=255, random_state=rnd, n_jobs=-1, verbosity=-1,
                    deterministic=True, force_col_wise=True, importance_type="gain",
                )
                model.fit(
                    LGB_TRAIN.iloc[tr_idx], y[tr_idx],
                    eval_set=[(LGB_TRAIN.iloc[va_idx], y[va_idx])],
                    eval_metric="auc", categorical_feature=CAT_COLS,
                    callbacks=[early_stopping(EARLY_STOP, verbose=False), log_evaluation(0)],
                )
                seed_oof[si] = model.predict_proba(LGB_TRAIN.iloc[va_idx])[:, 1]
                seed_test[si] = model.predict_proba(LGB_TEST)[:, 1]
                iter_info.append(model.best_iteration_)
                feats = list(LGB_TRAIN.columns)

            elif mname == "LGB_Deep":
                model = LGBMClassifier(
                    objective="binary", metric="auc",
                    n_estimators=2600 if not FAST_MODE else 250,
                    learning_rate=0.015, num_leaves=255, max_depth=-1,
                    min_child_samples=80, subsample=0.85, subsample_freq=1,
                    colsample_bytree=0.70, reg_alpha=0.50, reg_lambda=3.0,
                    max_bin=255, random_state=rnd + 7, n_jobs=-1, verbosity=-1,
                    deterministic=True, force_col_wise=True, importance_type="gain",
                )
                model.fit(
                    LGB_TRAIN.iloc[tr_idx], y[tr_idx],
                    eval_set=[(LGB_TRAIN.iloc[va_idx], y[va_idx])],
                    eval_metric="auc", categorical_feature=CAT_COLS,
                    callbacks=[early_stopping(EARLY_STOP, verbose=False), log_evaluation(0)],
                )
                seed_oof[si] = model.predict_proba(LGB_TRAIN.iloc[va_idx])[:, 1]
                seed_test[si] = model.predict_proba(LGB_TEST)[:, 1]
                iter_info.append(model.best_iteration_)
                feats = list(LGB_TRAIN.columns)

            elif mname == "XGB":
                model = XGBClassifier(
                    objective="binary:logistic", eval_metric="auc",
                    n_estimators=2200 if not FAST_MODE else 250,
                    learning_rate=0.03, max_depth=7, min_child_weight=6.0,
                    subsample=0.90, colsample_bytree=0.90, reg_alpha=0.05,
                    reg_lambda=2.0, gamma=0.0, max_bin=256, tree_method="hist",
                    device="cpu", random_state=rnd, n_jobs=-1,
                    early_stopping_rounds=EARLY_STOP,
                )
                model.fit(
                    XGB_TRAIN.iloc[tr_idx], y[tr_idx],
                    eval_set=[(XGB_TRAIN.iloc[va_idx], y[va_idx])],
                    verbose=False,
                )
                seed_oof[si] = model.predict_proba(XGB_TRAIN.iloc[va_idx])[:, 1]
                seed_test[si] = model.predict_proba(XGB_TEST)[:, 1]
                iter_info.append(getattr(model, "best_iteration", None))
                feats = list(XGB_TRAIN.columns)

            elif mname == "CAT":
                model = CatBoostClassifier(
                    iterations=1500 if not FAST_MODE else 220,
                    depth=7, learning_rate=0.04, loss_function="Logloss",
                    eval_metric="AUC", random_seed=rnd, l2_leaf_reg=6.0,
                    random_strength=0.25, bootstrap_type="Bernoulli", subsample=0.90,
                    thread_count=-1, verbose=False, allow_writing_files=False,
                )
                model.fit(
                    CAT_TRAIN.iloc[tr_idx], y[tr_idx],
                    cat_features=CAT_COLS,
                    eval_set=(CAT_TRAIN.iloc[va_idx], y[va_idx]),
                    use_best_model=True, early_stopping_rounds=EARLY_STOP,
                    verbose=False,
                )
                seed_oof[si] = model.predict_proba(CAT_TRAIN.iloc[va_idx])[:, 1]
                seed_test[si] = model.predict_proba(CAT_TEST)[:, 1]
                iter_info.append(model.get_best_iteration())
                feats = list(CAT_TRAIN.columns)

            elif mname == "HGB":
                model = HistGradientBoostingClassifier(
                    max_iter=1200 if not FAST_MODE else 220,
                    learning_rate=0.04, max_leaf_nodes=31,
                    min_samples_leaf=30, l2_regularization=1.0,
                    early_stopping=True, validation_fraction=0.10,
                    n_iter_no_change=EARLY_STOP, random_state=rnd,
                )
                model.fit(XGB_TRAIN.iloc[tr_idx], y[tr_idx])
                seed_oof[si] = model.predict_proba(XGB_TRAIN.iloc[va_idx])[:, 1]
                seed_test[si] = model.predict_proba(XGB_TEST)[:, 1]
                iter_info.append(getattr(model, "n_iter_", None))
                feats = list(XGB_TRAIN.columns)

            elif mname == "LGB_EXT":
                aug_x = pd.concat([LGB_TRAIN.iloc[tr_idx], LGB_ORIG], axis=0, ignore_index=True)
                aug_y = np.concatenate([y[tr_idx], y_original])
                model = LGBMClassifier(
                    objective="binary", metric="auc",
                    n_estimators=2200 if not FAST_MODE else 250,
                    learning_rate=0.03, num_leaves=63, max_depth=-1,
                    min_child_samples=80, subsample=0.90, subsample_freq=1,
                    colsample_bytree=0.90, reg_alpha=0.15, reg_lambda=1.50,
                    max_bin=255, random_state=rnd, n_jobs=-1, verbosity=-1,
                    deterministic=True, force_col_wise=True, importance_type="gain",
                )
                model.fit(
                    aug_x, aug_y,
                    eval_set=[(LGB_TRAIN.iloc[va_idx], y[va_idx])],
                    eval_metric="auc", categorical_feature=CAT_COLS,
                    callbacks=[early_stopping(EARLY_STOP, verbose=False), log_evaluation(0)],
                )
                seed_oof[si] = model.predict_proba(LGB_TRAIN.iloc[va_idx])[:, 1]
                seed_test[si] = model.predict_proba(LGB_TEST)[:, 1]
                iter_info.append(model.best_iteration_)
                feats = list(LGB_TRAIN.columns)
                del aug_x, aug_y

            if si == 0:
                try:
                    imp = model.feature_importances_
                except Exception:
                    imp = np.zeros(len(feats))
                for f, v in zip(feats, imp):
                    importance_rows.append((mname, fold, f, float(v)))

            del model
            gc.collect()

        oof[mname][va_idx] = seed_oof.mean(axis=0)
        test_fold_preds[mname][fold - 1] = seed_test.mean(axis=0)
        fold_auc = roc_auc_score(y[va_idx], oof[mname][va_idx])
        print(f"{mname:<9} AUC: {fold_auc:.6f} | seeds={n_seeds} | iter={iter_info}")

    gc.collect()

training_seconds = time.time() - fit_start
test_pred = {m: test_fold_preds[m].mean(axis=0) for m in MODEL_NAMES}

assert (fold_id >= 0).all()
assert all(np.isfinite(oof[m]).all() for m in MODEL_NAMES)
assert all(np.isfinite(test_pred[m]).all() for m in MODEL_NAMES)

banner("BASE OOF COMPLETE")
print(f"Training time: {training_seconds / 60:.1f} min")


In [ ]:
# 8. Base-model OOF report
model_scores = {}
model_fold_scores = {}

banner("TRUE OOF SCORES")
for m in MODEL_NAMES:
    global_auc = roc_auc_score(y, oof[m])
    folds = [
        roc_auc_score(y[fold_id == f], oof[m][fold_id == f])
        for f in range(N_FOLDS)
    ]
    model_scores[m] = global_auc
    model_fold_scores[m] = folds
    print(f"{m:<9} OOF={global_auc:.6f} | folds={[round(v, 6) for v in folds]}")

best_single = max(model_scores, key=model_scores.get)
print(f"\nBest single model: {best_single} ({model_scores[best_single]:.6f})")


In [ ]:
# 9. Nested OOF ensemble selection (prob / rank / scipy / prob+rank)
model_order = MODEL_NAMES
OOF_MATRIX = np.column_stack([oof[m] for m in model_order]).astype(np.float64)
TEST_MATRIX = np.column_stack([test_pred[m] for m in model_order]).astype(np.float64)

def transform_matrix(M, mode):
    if mode in ("prob", "scipy"):
        return M
    if mode == "rank":
        R = np.empty_like(M, dtype=np.float64)
        n = len(M)
        for j in range(M.shape[1]):
            R[:, j] = rankdata(M[:, j], method="average") / (n + 1.0)
        return R
    raise ValueError(mode)

def greedy_auc_weights(M, yy, passes=2):
    n_models = M.shape[1]
    single_auc = [roc_auc_score(yy, M[:, j]) for j in range(n_models)]
    start = int(np.argmax(single_auc))
    weights = np.zeros(n_models, dtype=np.float64)
    weights[start] = 1.0
    current = M[:, start].copy()
    current_auc = single_auc[start]

    grid = np.linspace(0.0, 0.60, 25)
    for _ in range(passes):
        improved = False
        for j in range(n_models):
            local_best_auc = current_auc
            local_best_alpha = 0.0
            for alpha in grid[1:]:
                cand = (1.0 - alpha) * current + alpha * M[:, j]
                a = roc_auc_score(yy, cand)
                if a > local_best_auc + 1e-7:
                    local_best_auc = a
                    local_best_alpha = alpha
            if local_best_alpha > 0:
                weights *= (1.0 - local_best_alpha)
                weights[j] += local_best_alpha
                current = M @ weights
                current_auc = local_best_auc
                improved = True
        if not improved:
            break

    weights /= weights.sum()
    return weights, current_auc

def scipy_auc_weights(M, yy, restarts=2, maxiter=150, subsample=200000):
    n = M.shape[1]
    rng = np.random.RandomState(SEED)
    take = min(len(M), subsample)
    idx = rng.choice(len(M), take, replace=False)
    Ms, ys = M[idx], yy[idx]

    def neg_auc(w):
        w = np.abs(w)
        w = w / (w.sum() + 1e-12)
        return -roc_auc_score(ys, Ms @ w)

    best_w, best_auc = None, -1.0
    for _ in range(restarts):
        x0 = rng.dirichlet(np.ones(n))
        res = minimize(neg_auc, x0, method="Nelder-Mead",
                       options={"maxiter": maxiter, "xatol": 1e-6, "fatol": 1e-7})
        w = np.abs(res.x)
        w = w / (w.sum() + 1e-12)
        a = roc_auc_score(ys, Ms @ w)
        if a > best_auc:
            best_auc, best_w = a, w
    return best_w, best_auc

BLEND_MODES = ["prob", "rank", "scipy"]
nested_pred = {m: np.zeros(n_train, dtype=np.float64) for m in BLEND_MODES}
nested_weights = {m: [] for m in BLEND_MODES}

for meta_fold in range(N_FOLDS):
    fit_mask = fold_id != meta_fold
    val_mask = fold_id == meta_fold

    for mode in BLEND_MODES:
        M_fit = transform_matrix(OOF_MATRIX[fit_mask], mode)
        M_val = transform_matrix(OOF_MATRIX[val_mask], mode)

        if mode == "scipy":
            w, _ = scipy_auc_weights(M_fit, y[fit_mask])
        else:
            w, _ = greedy_auc_weights(M_fit, y[fit_mask])
        nested_pred[mode][val_mask] = M_val @ w
        nested_weights[mode].append(w)

# Blend-of-blends: prob + rank average.
nested_pred["probrank"] = 0.5 * (nested_pred["prob"] + nested_pred["rank"])

all_modes = BLEND_MODES + ["probrank"]
nested_scores = {mode: roc_auc_score(y, nested_pred[mode]) for mode in all_modes}
nested_fold_scores = {
    mode: [roc_auc_score(y[fold_id == f], nested_pred[mode][fold_id == f]) for f in range(N_FOLDS)]
    for mode in all_modes
}

banner("NESTED ENSEMBLE AUDIT")
for mode in all_modes:
    print(f"{mode.upper():<9} nested OOF={nested_scores[mode]:.6f} | folds={[round(v, 6) for v in nested_fold_scores[mode]]}")

best_blend_mode = max(nested_scores, key=nested_scores.get)
best_blend_auc = nested_scores[best_blend_mode]
best_single_auc = model_scores[best_single]

# Select final strategy using honest OOF evidence.
if best_blend_auc > best_single_auc + 1e-5:
    FINAL_STRATEGY = f"{best_blend_mode.upper()}_BLEND"
    FINAL_OOF = nested_pred[best_blend_mode].copy()
    FINAL_OOF_AUC = best_blend_auc
    FINAL_FOLD_SCORES = nested_fold_scores[best_blend_mode]

    if best_blend_mode == "probrank":
        w_prob, _ = greedy_auc_weights(transform_matrix(OOF_MATRIX, "prob"), y)
        w_rank, _ = greedy_auc_weights(transform_matrix(OOF_MATRIX, "rank"), y)
        FINAL_WEIGHTS = 0.5 * w_prob + 0.5 * w_rank
        FINAL_TEST = 0.5 * (transform_matrix(TEST_MATRIX, "prob") @ w_prob) \
                   + 0.5 * (transform_matrix(TEST_MATRIX, "rank") @ w_rank)
    else:
        M_full = transform_matrix(OOF_MATRIX, best_blend_mode)
        if best_blend_mode == "scipy":
            FINAL_WEIGHTS, _ = scipy_auc_weights(M_full, y)
        else:
            FINAL_WEIGHTS, _ = greedy_auc_weights(M_full, y)
        M_test = transform_matrix(TEST_MATRIX, best_blend_mode)
        FINAL_TEST = M_test @ FINAL_WEIGHTS
else:
    FINAL_STRATEGY = best_single
    FINAL_OOF = oof[best_single].astype(np.float64)
    FINAL_OOF_AUC = best_single_auc
    FINAL_FOLD_SCORES = model_fold_scores[best_single]
    FINAL_WEIGHTS = np.zeros(len(model_order))
    FINAL_WEIGHTS[model_order.index(best_single)] = 1.0
    FINAL_TEST = test_pred[best_single].astype(np.float64)

print(f"\nSelected strategy : {FINAL_STRATEGY}")
print(f"Trusted OOF AUC   : {FINAL_OOF_AUC:.6f}")
print("Final test weights:")
for m, w in zip(model_order, FINAL_WEIGHTS):
    print(f"  * {m:<9} {w:.4f}")


In [ ]:
# 10. Diagnostics: diversity + feature importance
banner("COMPLEMENTARITY")
corr = pd.DataFrame(OOF_MATRIX, columns=model_order).corr(method="spearman")
print(corr.round(4).to_string())

imp_df = pd.DataFrame(importance_rows, columns=["model", "fold", "feature", "importance"])
imp_summary = (
    imp_df.groupby(["model", "feature"], as_index=False)["importance"].mean()
          .sort_values(["model", "importance"], ascending=[True, False])
)

print("\nTop features per model:")
for m in imp_summary["model"].unique():
    top = imp_summary[imp_summary["model"] == m].head(8)
    print(f"\n{m}")
    for row in top.itertuples():
        print(f"  * {row.feature:<34} {row.importance:.3f}")


In [ ]:
# 11. Save OOF + submission
pred_by_id = pd.DataFrame({
    ID_COL: test[ID_COL].to_numpy(),
    TARGET: np.clip(FINAL_TEST, 0.0, 1.0),
})

submission = sample[[ID_COL]].merge(pred_by_id, on=ID_COL, how="left")
assert submission[TARGET].notna().all()
assert submission.shape == sample.shape
assert submission[ID_COL].equals(sample[ID_COL])

submission.to_csv("submission.csv", index=False, float_format="%.10f")

oof_out = train[[ID_COL, TARGET]].copy()
oof_out["fold"] = fold_id
for m in model_order:
    oof_out[f"oof_{m.lower()}"] = oof[m]
oof_out["oof_final"] = FINAL_OOF
oof_out.to_csv("oof_predictions.csv", index=False, float_format="%.10f")

submission_sha256 = hashlib.sha256(Path("submission.csv").read_bytes()).hexdigest()

banner("FILES WRITTEN")
print(f"submission.csv       {submission.shape}")
print(f"oof_predictions.csv  {oof_out.shape}")
print(f"submission SHA256    {submission_sha256}")


In [ ]:
# 12. Final audit
decision_log = [
    "Used stratified K-fold OOF on the full competition training set.",
    "Excluded id from all model features.",
    "Feature engineering is deterministic and target-independent.",
    "All base-model scores come from true OOF predictions on identical folds.",
    "Ensemble selection uses nested OOF; test predictions never choose weights.",
    "CPU deterministic settings are used for reproducibility.",
    "Seed bagging (2 seeds for LGB/XGB/HGB) reduces OOF variance.",
    "Extended blend search covers prob, rank, scipy and prob+rank modes.",
]

if HAS_ORIGINAL:
    decision_log += [
        "Original public smartphone data detected and used as an external challenger.",
        "Severity teacher was trained only on original public rows.",
        "Original-row teacher features are OOF predictions, preventing teacher self-fit leakage.",
        "LGB_EXT adds original labeled rows only to each competition training fold.",
    ]
else:
    decision_log += [
        "Original public dataset was not mounted; teacher and LGB_EXT were skipped.",
    ]

banner("FINAL AUDIT")
print(f"FINAL OOF ROC AUC: {FINAL_OOF_AUC:.6f}")
print(f"Fold scores: {[round(v, 6) for v in FINAL_FOLD_SCORES]}")
print(f"Models used: {model_order}")
print(f"Selected strategy: {FINAL_STRATEGY}")
print(f"Number of engineered features: {X_train.shape[1]}")
print(f"Training time: {training_seconds / 60:.1f} minutes")
print(f"Submission shape: {submission.shape}")
print("Submission file: submission.csv")

print("\nDecision log:")
for i, item in enumerate(decision_log, 1):
    print(f"  {i}. {item}")

print("\nOOF first. Signal only. No leaderboard mythology.")
